# 05 — Deep Learning: MLP & FT-Transformer

In [1]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
import numpy as np, pandas as pd, torch, mlflow
from src.data import load_processed
from src.models.mlp import MLPRegressor
from src.models.ft_transformer import FTTransformer
from src.train import TrainConfig, train_torch_model, predict_torch
from src.evaluate import regression_metrics
from src.utils import set_seed
from src.config import MLFLOW_URI, EXPERIMENT_NAME, MODELS_DIR
set_seed(42); mlflow.set_tracking_uri(MLFLOW_URI); mlflow.set_experiment(EXPERIMENT_NAME)

<Experiment: artifact_location=('file:///D:/ZE5 PORTOFOLIO DS/Q-Factor Prediction in Optical Communication '
 'Systems/mlruns/622530031528693603'), creation_time=1777606059837, experiment_id='622530031528693603', last_update_time=1777606059837, lifecycle_stage='active', name='qfactor_prediction', tags={}, trace_location=None, workspace='default'>

In [2]:
X_train, y_train = load_processed('train')
X_val, y_val = load_processed('val')
input_dim = X_train.shape[1]; print('input_dim =', input_dim, 'device:', 'cuda' if torch.cuda.is_available() else 'cpu')

input_dim = 5 device: cpu


## MLP

In [3]:
mlp = MLPRegressor(input_dim, hidden_dims=(128, 128, 64), dropout=0.1)
cfg = TrainConfig(epochs=20, batch_size=8192, lr=1e-3)
with mlflow.start_run(run_name='MLP'):
    res = train_torch_model(mlp, X_train, y_train, X_val, y_val, cfg)
    preds = predict_torch(res['model'], X_val)
    m = regression_metrics(y_val, preds); print(m); mlflow.log_metrics(m)
torch.save(res['model'].state_dict(), MODELS_DIR / 'mlp.pt')

  ep 01/20 *  train=6.5035  val=0.1225  best=0.1225  (17.9s)


  ep 02/20 *  train=0.2902  val=0.0205  best=0.0205  (17.4s)


  ep 03/20 *  train=0.2261  val=0.0159  best=0.0159  (17.3s)


  ep 04/20 *  train=0.1946  val=0.0151  best=0.0151  (16.9s)


  ep 05/20 *  train=0.1704  val=0.0145  best=0.0145  (18.6s)


...

  ep 16/20 *  train=0.1191  val=0.0122  best=0.0122  (16.7s)


  ep 17/20    train=0.1187  val=0.0123  best=0.0122  (15.9s)


  ep 18/20 *  train=0.1180  val=0.0121  best=0.0121  (16.5s)


  ep 19/20    train=0.1179  val=0.0122  best=0.0121  (16.2s)


  ep 20/20    train=0.1181  val=0.0122  best=0.0121  (16.6s)
{'RMSE': 0.10979707630448991, 'MAE': 0.08717946708202362, 'R2': 0.9911346435546875, 'MAPE': 1.7303409576416016}


## FT-Transformer

In [4]:
ftt = FTTransformer(input_dim, d_token=64, n_blocks=3, n_heads=8, dropout=0.1)
cfg = TrainConfig(epochs=15, batch_size=4096, lr=5e-4)
with mlflow.start_run(run_name='FT-Transformer'):
    res = train_torch_model(ftt, X_train, y_train, X_val, y_val, cfg)
    preds = predict_torch(res['model'], X_val)
    m = regression_metrics(y_val, preds); print(m); mlflow.log_metrics(m)
torch.save(res['model'].state_dict(), MODELS_DIR / 'ft_transformer.pt')

  ep 01/15 *  train=1.5962  val=0.1409  best=0.1409  (174.6s)


  ep 02/15 *  train=0.1164  val=0.0425  best=0.0425  (163.9s)


  ep 03/15 *  train=0.0598  val=0.0289  best=0.0289  (170.3s)


  ep 04/15 *  train=0.0444  val=0.0239  best=0.0239  (165.9s)


  ep 05/15 *  train=0.0371  val=0.0222  best=0.0222  (168.5s)


...

  ep 11/15    train=0.0255  val=0.0194  best=0.0176  (200.1s)


  ep 12/15    train=0.0250  val=0.0180  best=0.0176  (225.0s)


  ep 13/15    train=0.0247  val=0.0178  best=0.0176  (248.9s)


  ep 14/15    train=0.0246  val=0.0181  best=0.0176  (222.5s)


  ep 15/15    train=0.0244  val=0.0177  best=0.0176  (217.4s)
  early-stop at epoch 15 (no val improvement for 5 epochs)
{'RMSE': 0.13259660479636803, 'MAE': 0.10596104711294174, 'R2': 0.9870705604553223, 'MAPE': 2.1097934246063232}
